In [2]:
pip install langchain langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 10.8 MB/s eta 0:00:00


In [3]:
import json
import os
from datetime import datetime
from typing import Dict, TypedDict, Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

In [7]:
# Load environment
from dotenv import load_dotenv
os.environ["GROQ_API_KEY"] = "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"

MODEL_NAME = "llama-3.3-70b-versatile"



In [8]:
class ReportGenerationState(TypedDict):
    """State for Agent 3 - Report Generation"""
    agent2_analysis: str  # Analysis from Agent 2
    agent2_summary: str  # Summary from Agent 2
    user_preferences: Dict  # User preferences
    final_report: Optional[str]  # Generated comprehensive report
    report_metadata: Optional[Dict]  # Metadata about the report

In [9]:
# ============================================
# AGENT 3: REPORT GENERATOR NODE
# ============================================

def generate_comprehensive_report(state: ReportGenerationState) -> ReportGenerationState:
    """
    AGENT 3: Generate final comprehensive PDF-style report

    Takes Agent 2's analysis and summary, creates a professional
    real estate recommendation report with formatting
    """

    print("\n" + "="*80)
    print("📝 AGENT 3: GENERATING COMPREHENSIVE RECOMMENDATION REPORT")
    print("="*80)

    # Get input from Agent 2
    analysis = state.get("agent2_analysis", "")
    summary = state.get("agent2_summary", "")
    user_prefs = state.get("user_preferences", {})

    # Validation
    if not analysis or not summary:
        print("❌ Error: Missing analysis or summary from Agent 2")
        return {
            "final_report": "Error: Insufficient data from Agent 2",
            "report_metadata": {"status": "failed"}
        }

    print(f"✓ Received analysis from Agent 2")
    print(f"✓ Received summary from Agent 2")
    print(f"✓ User preferences loaded")

    # Initialize LLM
    print(f"\n🤖 Initializing LLM ({MODEL_NAME})...")
    try:
        llm = ChatGroq(model=MODEL_NAME, temperature=0.2)
        print("   ✓ LLM initialized")
    except Exception as e:
        print(f"   ❌ LLM initialization failed: {e}")
        return {
            "final_report": f"Error: Could not initialize LLM - {str(e)}",
            "report_metadata": {"status": "failed"}
        }

    # Build comprehensive prompt
    print(f"\n✍️  Generating comprehensive report...")

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a professional real estate advisor creating a final recommendation report for a home buyer.

Your task is to transform the analysis and summary into a comprehensive, beautifully formatted report that looks professional and is ready for PDF export.

CRITICAL FORMATTING REQUIREMENTS:
- Use clear section headers with ═══ separators
- Use bullet points with • or ✓ symbols
- Use emojis sparingly (🏠, ✓, ⚠, 🏆, 📊, 💰, 🎯)
- Number your rankings clearly
- Make it look like a professional report document
- Be specific with data and numbers
- Write in a confident, advisory tone"""),

        ("user", """Create a comprehensive home buying recommendation report using this information:

PROPERTY ANALYSIS FROM AGENT 2:
{analysis}

SUMMARY & RANKINGS FROM AGENT 2:
{summary}

USER PREFERENCES:
{preferences}

---

CREATE A COMPLETE REPORT WITH THESE EXACT SECTIONS:

═══════════════════════════════════════════════════════════════
🏠 HOME BUYING RECOMMENDATION REPORT
═══════════════════════════════════════════════════════════════

**Report Generated:** {date}
**Prepared For:** Home Buyer
**Location:** North Carolina
**Budget:** Under $450,000

═══════════════════════════════════════════════════════════════
EXECUTIVE SUMMARY
═══════════════════════════════════════════════════════════════

Write a 4-5 sentence executive summary that:
- States which property is recommended (use the #1 ranked property from summary)
- Explains WHY it's the best choice based on user's priorities
- Mentions the rating (e.g., "8 out of 10")
- Highlights 2-3 key strengths
- Addresses any important trade-offs

═══════════════════════════════════════════════════════════════
DETAILED PROPERTY ANALYSIS
═══════════════════════════════════════════════════════════════

For EACH of the 3 properties, create a detailed breakdown using this format:

---
### Property #[rank]: [Property Name/Location]
**Overall Rating: [X]/10** ⭐⭐⭐⭐

**PROPERTY DETAILS**
- List Price: $[price]
- Bedrooms: [#] beds
- Bathrooms: [#] baths
- Neighborhood: [name]
- Commute to Uptown: [time]

**LOCATION QUALITY METRICS**
- Crime Rate: [rating with description]
- School District: [rating]/10
- [Other relevant details from analysis]

**STRENGTHS** ✓
[List each strength from Agent 2's analysis with ✓ bullet points]

**WEAKNESSES** ⚠
[List each weakness from Agent 2's analysis with ⚠ bullet points]

**KEY CONSIDERATIONS**
[List key considerations from Agent 2's analysis as bullet points]

**BEST SUITED FOR**
[Type of buyer this property suits based on the analysis]

---

[Repeat this format for all 3 properties]

═══════════════════════════════════════════════════════════════
COMPARATIVE ANALYSIS
═══════════════════════════════════════════════════════════════

Create a comparison table showing:

| Factor | Property 1 | Property 2 | Property 3 |
|--------|-----------|-----------|-----------|
| Price | [price] | [price] | [price] |
| Rating | [X]/10 | [X]/10 | [X]/10 |
| Crime Rate | [grade] | [grade] | [grade] |
| Schools | [rating]/10 | [rating]/10 | [rating]/10 |
| Commute | [time] | [time] | [time] |

═══════════════════════════════════════════════════════════════
🏆 FINAL RECOMMENDATION
═══════════════════════════════════════════════════════════════

**RECOMMENDED PROPERTY: Property #1 - [Name]**

Write 3-4 paragraphs that:
1. Clearly state this is your #1 recommendation
2. Explain WHY based on user's stated priorities (safety, schools, budget)
3. Address how it outperforms the other options
4. Acknowledge any trade-offs (HOA fees, price, etc.)
5. Provide confidence in the recommendation
6. Mention long-term value and resale considerations

**CONFIDENCE LEVEL:** [High/Medium/Low]
**RISK ASSESSMENT:** [Low/Medium/High]

═══════════════════════════════════════════════════════════════
📊 FINANCIAL BREAKDOWN - RECOMMENDED PROPERTY
═══════════════════════════════════════════════════════════════

For the recommended property, provide estimated financial details:

**Purchase Details**
- List Price: $[price]
- Estimated Down Payment (20%): $[amount]
- Estimated Loan Amount: $[amount]

**Estimated Monthly Costs**
- Principal & Interest (7% APR, 30yr): ~$[amount]
- Property Tax (est. 1.2% annually): ~$[amount]/month
- Home Insurance (est.): ~$150/month
- HOA Fees: $[amount]/month
- **Total Estimated Monthly Cost: $[total]**

**Long-Term Projection**
- Total 5-Year Cost (down payment + 60 months): ~$[amount]
- Estimated Home Value in 5 Years (6% appreciation): ~$[amount]

═══════════════════════════════════════════════════════════════
🎯 NEXT STEPS - ACTION PLAN
═══════════════════════════════════════════════════════════════

Provide 6 specific, actionable next steps:

**IMMEDIATE ACTIONS (This Week)**

1. ✅ **Schedule Property Showing**
   [Specific guidance about viewing the property]

2. ✅ **Get Pre-Approved for Mortgage**
   [Specific guidance about mortgage pre-approval]

**SHORT-TERM ACTIONS (Next 2-4 Weeks)**

3. ✅ **Conduct Professional Home Inspection**
   [Specific guidance about home inspection]

4. ✅ **Review HOA Documents**
   [Specific guidance about HOA review]

5. ✅ **Make Purchase Offer**
   [Specific guidance about making offer, suggested offer price]

6. ✅ **Plan for Closing**
   [Specific guidance about closing timeline and requirements]

═══════════════════════════════════════════════════════════════
IMPORTANT DISCLAIMERS
═══════════════════════════════════════════════════════════════

- This report is based on publicly available information and AI analysis
- Property prices and availability subject to change
- Professional inspection and appraisal recommended
- Consult with real estate attorney before purchase
- Financial estimates are approximate

═══════════════════════════════════════════════════════════════

**Report Prepared By:** AI Real Estate Assistant
**Date:** {date}
**Valid For:** 30 days

═══════════════════════════════════════════════════════════════

IMPORTANT: Use the exact information from the analysis and summary. Do not make up new data. Be specific and use all the details provided.""")
    ])

    # Format user preferences
    prefs_text = "\n".join([f"• {k}: {v}" for k, v in user_prefs.items()])

    # Get current date
    current_date = datetime.now().strftime("%B %d, %Y")

    # Generate report
    try:
        chain = prompt | llm

        response = chain.invoke({
            "analysis": analysis,
            "summary": summary,
            "preferences": prefs_text,
            "date": current_date
        })

        report_text = response.content

        print("   ✓ Report generated successfully")

        # Create metadata
        metadata = {
            "status": "success",
            "generated_date": current_date,
            "num_properties_analyzed": 3,
            "report_length_chars": len(report_text),
            "recommended_property": extract_recommended_property(summary)
        }

        print(f"\n✅ Report Complete:")
        print(f"   Recommended Property: {metadata['recommended_property']}")
        print(f"   Report Length: {metadata['report_length_chars']} characters")

    except Exception as e:
        print(f"   ❌ Error generating report: {e}")
        return {
            "final_report": f"Error generating report: {str(e)}",
            "report_metadata": {"status": "failed", "error": str(e)}
        }

    print("\n" + "="*80)
    print("✅ AGENT 3 COMPLETE")
    print("="*80)

    return {
        "final_report": report_text,
        "report_metadata": metadata
    }


In [10]:
# ============================================
# HELPER FUNCTIONS
# ============================================

def extract_recommended_property(summary: str) -> str:
    """Extract the recommended property name from summary"""

    # Look for "Property 1:" or similar patterns
    lines = summary.split('\n')
    for line in lines:
        if "Property 1:" in line or "**Property 1:" in line:
            # Extract property name
            if ":" in line:
                parts = line.split(":")
                if len(parts) >= 2:
                    prop_name = parts[1].split("**")[0].split("(")[0].strip()
                    return prop_name

    return "Property 1"


def display_report(result: Dict):
    """
    Display the generated report in a formatted way
    """

    print("\n" + "="*80)
    print("📄 FINAL RECOMMENDATION REPORT")
    print("="*80 + "\n")

    print(result["final_report"])

    print("\n" + "="*80)
    print("📊 REPORT METADATA")
    print("="*80)

    metadata = result.get("report_metadata", {})
    print(f"Status: {metadata.get('status', 'N/A')}")
    print(f"Generated: {metadata.get('generated_date', 'N/A')}")
    print(f"Recommended Property: {metadata.get('recommended_property', 'N/A')}")
    print(f"Properties Analyzed: {metadata.get('num_properties_analyzed', 'N/A')}")

    print("="*80 + "\n")


def save_report_to_file(result: Dict, filename: str = "home_recommendation_report.txt"):
    """
    Save the report to a text file
    """

    try:
        full_path = os.path.abspath(filename)

        with open(full_path, 'w', encoding='utf-8') as f:
            # Write main report
            f.write(result["final_report"])

            # Add metadata footer
            f.write("\n\n" + "="*80 + "\n")
            f.write("REPORT METADATA\n")
            f.write("="*80 + "\n")

            metadata = result.get("report_metadata", {})
            f.write(f"Status: {metadata.get('status', 'N/A')}\n")
            f.write(f"Generated: {metadata.get('generated_date', 'N/A')}\n")
            f.write(f"Recommended Property: {metadata.get('recommended_property', 'N/A')}\n")
            f.write(f"Properties Analyzed: {metadata.get('num_properties_analyzed', 'N/A')}\n")

        print(f"💾 Report saved to: {full_path}")
        return True

    except Exception as e:
        print(f"❌ Error saving report: {e}")
        return False


In [11]:
# ============================================
# BUILD AGENT 3 WORKFLOW
# ============================================

def build_agent3_workflow():
    """Build the Agent 3 LangGraph workflow"""

    from langgraph.graph import StateGraph, END

    # Create workflow
    agent3_workflow = StateGraph(ReportGenerationState)

    # Add node
    agent3_workflow.add_node("generate_report", generate_comprehensive_report)

    # Define flow
    agent3_workflow.set_entry_point("generate_report")
    agent3_workflow.add_edge("generate_report", END)

    # Compile
    return agent3_workflow.compile()


# ============================================
# INTEGRATION FUNCTION FOR TEAM
# ============================================

def run_agent3_with_agent2_output(agent2_result: Dict) -> Dict:
    """
    Main integration function - use this to connect Agent 3 with Agent 2

    Usage:
        # After Agent 2 completes
        agent2_output = agent2_app.invoke(...)

        # Pass to Agent 3
        agent3_output = run_agent3_with_agent2_output(agent2_output)

        # Display report
        display_report(agent3_output)
    """

    print("\n" + "🔗"*40)
    print("CONNECTING AGENT 2 OUTPUT TO AGENT 3")
    print("🔗"*40)

    # Build Agent 3 workflow
    agent3_app = build_agent3_workflow()

    # Prepare state for Agent 3
    agent3_state = {
        "agent2_analysis": agent2_result.get("analysis", ""),
        "agent2_summary": agent2_result.get("summary", ""),
        "user_preferences": agent2_result.get("user_preferences", {}),
        "final_report": None,
        "report_metadata": None
    }

    # Run Agent 3
    result = agent3_app.invoke(agent3_state)

    return result



In [12]:
# ============================================
# TEST FUNCTION WITH AGENT 2 SAMPLE OUTPUT
# ============================================

def test_agent3():
    """
    Test Agent 3 with the exact output from Agent 2
    """

    print("\n" + "🚀"*40)
    print("TESTING AGENT 3: REPORT GENERATION")
    print("🚀"*40 + "\n")

    # Check API key
    if not os.environ.get("GROQ_API_KEY"):
        print("\n❌ ERROR: GROQ_API_KEY not set!")
        print("\nPlease set your API key in .env file or:")
        print('os.environ["GROQ_API_KEY"] = "your-key-here"')
        return None

    # EXACT OUTPUT FROM AGENT 2 (from your notebook)
    agent2_sample_output = {
        "analysis": """### Property 1: Davidson

**Analysis:**
Strengths: Davidson offers a very low crime rate, excellent schools (9/10), and a desirable location near Lake Norman and a historic downtown. The property itself has 4 beds and 2.5 baths, which could be attractive for a family.
Weaknesses: The list price is $439,000, which is close to the budget limit. Additionally, the property has high HOA fees, which could be a deterrent for some buyers.

**Rating: 8/10**
This property is a strong contender due to its excellent safety rating, good schools, and desirable location. However, the high HOA fees and being near the budget limit are drawbacks.

**Key Considerations:**
- The high HOA fees need to be factored into the overall cost of ownership.
- The excellent school district and very low crime rate may justify the price and additional fees for buyers prioritizing safety and education.
- The commute time of 25-35 minutes to Uptown is reasonable but may not be ideal for those seeking a shorter commute.

### Property 2: South Charlotte (Piper Glen)

**Analysis:**
Strengths: This property is located in a low crime area, has a slightly shorter commute time (20-30 minutes), and backs onto a golf course, which could be a significant amenity for some buyers. The school district rating is 8/10, which is still considered good.
Weaknesses: The list price is $449,900, which is very close to the budget limit. The property has 3 beds and 3 baths, which might be less appealing for larger families compared to the other options.

**Rating: 7.5/10**
This property offers a good balance of safety, commute time, and school quality. However, being at the higher end of the budget and having fewer bedrooms than other options are notable drawbacks.

**Key Considerations:**
- The property's backing onto a golf course is a unique feature that may increase its appeal and potentially its resale value.
- The slightly shorter commute and low crime rate are positives, but the higher price point and fewer bedrooms might make it less desirable for some buyers.
- Buyers need to weigh the value of the golf course view and mature community against the higher price and the school district being slightly lower rated than Property 1.

### Property 3: East Charlotte (Oakhurst)

**Analysis:**
Strengths: The list price of $425,000 is more budget-friendly, and the commute time of 15-20 minutes is the shortest among the three properties. The property has been recently renovated, which could be a plus.
Weaknesses: The crime rate in the area is moderate, which is a significant concern given the buyer's very high priority on safety. The school district rating is 6/10, which is the lowest among the three properties and could impact resale value.

**Rating: 4/10**
Despite its budget-friendliness and shorter commute, the moderate crime rate and lower school district rating make this property less desirable given the buyer's priorities.

**Key Considerations:**
- The moderate crime rate and lower-rated school district are significant drawbacks that may outweigh the benefits of a shorter commute and lower price.
- Buyers who prioritize safety above all else may find this property unsuitable due to its crime rate.
- The recent renovation is a positive aspect, but it may not be enough to offset the concerns regarding safety and school quality.""",

        "summary": """**Summary and Recommendation:**

Based on the analysis, the properties can be ranked from best to worst as follows:

1. **Property 1: Davidson** (Rating: 8/10) - Offers excellent safety, good schools, and a desirable location, despite high HOA fees and a list price near the budget limit.
2. **Property 2: South Charlotte (Piper Glen)** (Rating: 7.5/10) - Provides a good balance of safety, commute time, and school quality, but is at the higher end of the budget and has fewer bedrooms.
3. **Property 3: East Charlotte (Oakhurst)** (Rating: 4/10) - Has a moderate crime rate and lower school district rating, making it less desirable despite its budget-friendliness and shorter commute.

**Recommendation:**
Given the buyer's high priority on safety and education, **Property 1: Davidson** is the top choice. Its excellent school district (9/10) and very low crime rate justify the higher list price and additional HOA fees. While it's near the budget limit, the benefits of safety, education, and location make it the most desirable option. Buyers should carefully consider the overall cost of ownership, including HOA fees, and weigh these against the property's significant strengths.""",

        "user_preferences": {
            "budget_priority": "High - want best value under $450k",
            "safety_priority": "Very High - low crime is essential",
            "commute_priority": "Medium - prefer under 30 min",
            "schools_priority": "High - good schools important for resale",
            "other_notes": "Willing to pay more for safety and schools"
        }
    }

    print("📥 Received output from Agent 2:")
    print(f"   - Analysis length: {len(agent2_sample_output['analysis'])} chars")
    print(f"   - Summary length: {len(agent2_sample_output['summary'])} chars")
    print(f"   - User preferences: {len(agent2_sample_output['user_preferences'])} items")

    # Run Agent 3
    result = run_agent3_with_agent2_output(agent2_sample_output)

    # Display report
    display_report(result)

    # Ask to save
    save = input("\n💾 Save report to file? (y/n): ").lower().strip()
    if save == 'y':
        save_report_to_file(result)

    print("\n" + "🎉"*40)
    print("AGENT 3 TEST COMPLETE!")
    print("🎉"*40 + "\n")

    return result



In [15]:
if __name__ == "__main__":

    print("\n" + "="*80)
    print("AGENT 3: RECOMMENDATION REPORT GENERATOR")
    print("="*80)

    # Run test
    result = test_agent3()

    if result:
        print("\n✅ Success! Agent 3 is ready to integrate with your team's workflow.")
        print("\n💡 To integrate with full workflow:")
        print("   agent3_output = run_agent3_with_agent2_output(agent2_result)")


AGENT 3: RECOMMENDATION REPORT GENERATOR

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
TESTING AGENT 3: REPORT GENERATION
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀

📥 Received output from Agent 2:
   - Analysis length: 3334 chars
   - Summary length: 1188 chars
   - User preferences: 5 items

🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗
CONNECTING AGENT 2 OUTPUT TO AGENT 3
🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗

📝 AGENT 3: GENERATING COMPREHENSIVE RECOMMENDATION REPORT
✓ Received analysis from Agent 2
✓ Received summary from Agent 2
✓ User preferences loaded

🤖 Initializing LLM (llama-3.3-70b-versatile)...
   ✓ LLM initialized

✍️  Generating comprehensive report...
   ✓ Report generated successfully

✅ Report Complete:
   Recommended Property: Davidson
   Report Length: 10049 characters

✅ AGENT 3 COMPLETE

📄 FINAL RECOMMENDATION REPORT

═══════════════════════════════════════════════════════════════
🏠 HOME BUYING RECOMMENDATION REPORT
═══════════════════════════════════════════════════════

In [14]:
%pip install langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 15.9 MB/s eta 0:00:00
